# Getting started with Cloud-Native HLS Data in Python:

Tutorial from: [LPDAAC](https://lpdaac.usgs.gov/documents/923/HLS_Tutorial_vRI80pO.html)

### Extracting an EVI Time Series from Harmonized Landsat-8 Sentinel-2 (HLS) data in the Cloud using CMR's SpatioTemporal Asset Catalog (CMR-STAC)
#### This tutorial demonstrates how to work with the HLS Landsat 8 (HLSL30.015) and Sentinel-2 (HLSS30.015) data products

The Harmonized Landsat and Sentinel-2 (HLS) project produces seamless, harmonized surface reflectance data from the Operational Land Imager (OLI) and Multi-Spectral Instrument (MSI) aboard Landsat-8 and Sentinel-2 Earth-observing satellites, respectively. The aim is to produce seamless products with normalized parameters, which include atmospheric correction, cloud and cloud-shadow masking, geographic co-registration and common gridding, normalized bidirectional reflectance distribution function, and spectral band adjustment. This will provide global observation of the Earth’s surface every 2-3 days with 30 meter spatial resolution. One of the major applications that will benefit from HLS is agriculture assessment and monitoring, which is used as the use case for this tutorial.

NASA's Land Processes Distributed Active Archive Center (LP DAAC) archives and distributes HLS products in the LP DAAC Cumulus cloud archive as Cloud Optimized GeoTIFFs (COG). This tutorial will demonstrate how to query and subset HLS data using the NASA Common Metadata Repository (CMR) SpatioTemporal Asset Catalog (STAC) application programming interface (API). Because these data are stored as COGs, this tutorial will teach users how to load subsets of individual files into memory for just the bands you are interested in--a paradigm shift from the more common workflow where you would need to download a .zip/HDF file containing every band over the entire scene/tile. This tutorial covers how to process HLS data (quality filtering and EVI calculation), visualize, and "stack" the scenes over a region of interest into an xarray data array, calculate statistics for an EVI time series, and export as a comma-separated values (CSV) file--providing you with all of the information you need for your area of interest without having to download the source data file. The Enhanced Vegetation Index (EVI), is a vegetation index similar to NDVI that has been found to be more sensitive to ground cover below the vegetated canopy and saturates less over areas of dense green vegetation.

## Use Case Example: Calculate mean EVI (Enhanced Vegetation Index) for a region of interest over time.

### What is EVI? 

The enhanced vetation index or EVI is a *next generation* greenness index that is designed to keep sensitivity in dense vegetation while reducing soil-background and aerosol influences that can bias simpler indicies like NDVI. It uses the blue band to correct for aerosol influences in the red band. The formula is:

$$
EVI = G * \frac{\left( NIR - Red \right)}{\left( NIR + C_1 * Red - C_2 * Blue + L \right)}
$$

where modis coefficients are used: $G = 2.5$, $C_1 = 6$, $C_2 = 7.5$, and $L = 1$.

In practice this means that EVI remains responsive when NDVI tends to saturate in high biomass regions. EVI better decouples vegetation signal from illumiation/atmosphere and soil background.

Greeness indexes like NDVI and EVI are widely used in remote sensing of vegetation to track phenology, monitor drought, estimate productivity, and more. They give you a single, dimensionless index that summarizes the spectral response of vegetation. 

### Why HLS data is ideal for EVI time series analysis:

- **Frequent, consistent revisit:** A virtual constellation gives ~2–3-day global coverage at 30 m, ideal for phenology, disturbance tracking, and rapid drought monitoring. 
NASA Earthdata

- **BRDF (view-angle) normalization to nadir:** Reduces angular artifacts that can otherwise leak into EVI, especially in mountainous areas. HLS v2 applies a c-factor BRDF adjustment and sets products to a nadir view. (BDRF = Bidirectional Reflectance Distribution Function)


- **Harmonized spectral response:** Sentinel-2 bands are bandpass-adjusted to match Landsat in the common bands, minimizing cross-sensor steps in your EVI series. (S30 provides OLI-like Blue/Red/NIR along with native red-edge bands.) 


- **Surface reflectance + solid QA:** Atmospheric correction uses LaSRC; cloud/shadow/snow/water masks come from Fmask (dilated), with aerosol level bits—useful for EVI quality screening. 


- **Ready-made VI products:** NASA now distributes an HLS-VI suite that includes EVI (and NDVI, SAVI, NDMI, etc.), so you can either compute from surface reflectance or ingest the precomputed EVI layers. 

- **Near-real-time latency:** Typical latency is about 1.7 days, making operational monitoring feasible.




**Data Used in the Example:**

- **PROVISIONAL daily 30 meter (m) global HLS Sentinel-2 Multi-spectral Instrument Surface Reflectance - HLSS30.015**

*The HLSS30 product provides 30 m Nadir normalized Bidirectional Reflectance Distribution Function (BRDF)-Adjusted Reflectance (NBAR) and is derived from Sentinel-2A and Sentinel-2B MSI data products.*
- **Science Dataset (SDS) layers:**
    - B8A (NIR Narrow)
    - B04 (Red)
    - B02 (Blue)
    - Fmask (Quality)
- **PROVISIONAL daily 30 meter (m) global HLS Landsat-8 OLI Surface Reflectance - HLSL30.015**

*The HLSL30 product provides 30 m Nadir normalized Bidirectional Reflectance Distribution Function (BRDF)-Adjusted Reflectance (NBAR) and is derived from Landsat-8 OLI data products.*
- **Science Dataset (SDS) layers:**
    - B05 (NIR)
    - B04 (Red)
    - B02 (Blue)
    - Fmask (Quality)

In [1]:
import os
from datetime import datetime
import requests as r
import numpy as np
import pandas as pd
import geopandas as gp
from skimage import io
import matplotlib.pyplot as plt
from osgeo import gdal
import rasterio as rio
from rasterio.mask import mask
from rasterio.enums import Resampling
from rasterio.shutil import copy
import pyproj
from pyproj import Proj
from shapely.ops import transform
import xarray as xr
import geoviews as gv
from cartopy import crs
import hvplot.xarray
import holoviews as hv
gv.extension('bokeh', 'matplotlib')

ModuleNotFoundError: No module named 'requests'

## References

This tutorial is adapted from the following resources:

Getting Started with Cloud-Native HLS Data in Python written by Cole Krehbiel1
Contact: LPDAAC@usgs.gov
Voice: +1-605-594-6116
Organization: Land Processes Distributed Active Archive Center (LP DAAC)
Website: https://lpdaac.usgs.gov/
Date last modified: 02-12-2021
1KBR Inc., contractor to the U.S. Geological Survey, Earth Resources Observation and Science (EROS) Center, Sioux Falls, South Dakota, 57198-001, USA. Work performed under USGS contract G15PD00467 for LP DAAC2. 2LP DAAC Work performed under NASA contract NNG14HH33I.